In [1]:
import os
import pandas as pd
from pathlib import Path
import shutil
from tqdm import tqdm
import random
import dask.dataframe as dd
from PIL import Image
import io

In [2]:
# load dataset specification files (based on Manifold Bias paper)

# calibration_real = pd.read_excel("calibration_real_paths.xlsx") -> not needed, as MS COCO 2017 is used for calibration
test_generated = pd.read_excel("test_generated_paths.xlsx")
test_real = pd.read_excel("test_real_paths.xlsx")

## CNNSpot
filtering relevant images from CNNSpot

### Real

In [7]:
CNNSpot_real = test_real[test_real["dataset name"] == "CNNSpot"].copy()

length_CNNSpot_real = len(CNNSpot_real)

length_CNNSpot_real

26088

In [15]:
CNNSpot_real.head()

,path,dataset name
3,train2/airplane/0_real/00928.png,CNNSpot
4,train2/cow/0_real/00758.png,CNNSpot
13,train2/sofa/0_real/00370.png,CNNSpot
14,train2/bird/0_real/00033.png,CNNSpot
20,train2/bus/0_real/01214.png,CNNSpot


In [16]:
split_CNNSpot_real = CNNSpot_real["path"].str.split(pat="/", expand=True)

# drop filename (last column)
split_CNNSpot_real = split_CNNSpot_real.iloc[:, :-1]

# optionally name the levels
split_CNNSpot_real.columns = [f"level_{i}" for i in range(split_CNNSpot_real.shape[1])]

# count per level and store in dict
level_counts = {
    col: split_CNNSpot_real[col].value_counts()
    for col in split_CNNSpot_real.columns
}

# print nicely sorted by level
for level, counts in level_counts.items():
    print(f"\n=== {level} ===")
    print(counts)


=== level_0 ===
level_0
train2     24920
backup2     1168
Name: count, dtype: int64

=== level_1 ===
level_1
bottle         1317
sofa           1312
person         1311
cat            1311
sheep          1310
car            1310
bicycle        1310
tvmonitor      1309
cow            1309
horse          1307
boat           1307
airplane       1304
train          1301
dog            1301
chair          1298
diningtable    1297
bird           1297
pottedplant    1296
motorbike      1292
bus            1289
Name: count, dtype: int64

=== level_2 ===
level_2
0_real    26088
Name: count, dtype: int64


In [17]:
splitted_CNNSpot_real = CNNSpot_real["path"].str.split(pat="/")
CNNSpot_real["real_path"] = ("CNNSpot/train/" + splitted_CNNSpot_real.str[1]+ "/" + splitted_CNNSpot_real.str[2]+ "/" + splitted_CNNSpot_real.str[3])

In [19]:
root = Path("../../datasets/")

CNNSpot_real["exists"] = [
    (root / p).exists() for p in CNNSpot_real["real_path"]
]

print(CNNSpot_real["exists"].value_counts()) # results mean that all file paths exist, within the CNNSpot folder

exists
True    26088
Name: count, dtype: int64


In [20]:
duplicates_CNN_real = CNNSpot_real.real_path[CNNSpot_real.real_path.duplicated()]
(f"Number of duplicate paths: {len(duplicates_CNN_real)}")

'Number of duplicate paths: 18'

In [21]:
src_root = Path("../../datasets")      # where files currently are
dst_root = Path("../../replication_datasets")   # where you want them

for rel_path in CNNSpot_real["real_path"]:
    src = src_root / rel_path
    dst = dst_root / rel_path

    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)  # create folders
        shutil.copy2(src, dst)  # preserves metadata
    else:
        print(f"Missing: {src}")

In [24]:
root = Path("../../replication_datasets/train")

extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}

count = sum(1 for p in root.rglob("*") if p.suffix.lower() in extensions)

print(count)
print(length_CNNSpot_real-18) # there are 18 duplicates

26070
26070


### Fake

In [3]:
CNNSpot_fake = test_generated[test_generated["dataset name"] == "CNNSpot"].copy()

length_CNNSpot_fake = len(CNNSpot_fake)

length_CNNSpot_fake

72590

In [4]:
CNNSpot_fake.head()

,path,dataset name
0,stylegan/car/1_fake/008017.png,CNNSpot
1,train2/person/1_fake/00570.png,CNNSpot
3,cyclegan/orange/1_fake/n07740461_12101_fake.png,CNNSpot
4,train2/cow/1_fake/01321.png,CNNSpot
5,cyclegan/orange/1_fake/n07740461_14960_fake.png,CNNSpot


In [5]:
split_fake = CNNSpot_fake["path"].str.split(pat="/")

CNNSpot_fake["part0"] = split_fake.str[0]
CNNSpot_fake["part1"] = split_fake.str[1]
CNNSpot_fake["part2"] = split_fake.str[2]
CNNSpot_fake["part3"] = split_fake.str[3]

In [6]:
CNNSpot_fake

,path,dataset name,part0,part1,part2,part3
0,stylegan/car/1_fake/008017.png,CNNSpot,stylegan,car,1_fake,008017.png
1,train2/person/1_fake/00570.png,CNNSpot,train2,person,1_fake,00570.png
3,cyclegan/orange/1_fake/n07740461_12101_fake.png,CNNSpot,cyclegan,orange,1_fake,n07740461_12101_fake.png
4,train2/cow/1_fake/01321.png,CNNSpot,train2,cow,1_fake,01321.png
5,cyclegan/orange/1_fake/n07740461_14960_fake.png,CNNSpot,cyclegan,orange,1_fake,n07740461_14960_fake.png
...,...,...,...,...,...,...
100694,stylegan2/cat/1_fake/001589.png,CNNSpot,stylegan2,cat,1_fake,001589.png
100695,stylegan2/horse/1_fake/001545.png,CNNSpot,stylegan2,horse,1_fake,001545.png
100696,train2/train/1_fake/00997.png,CNNSpot,train2,train,1_fake,00997.png
100698,crn/1_fake/100212_output.png,CNNSpot,crn,1_fake,100212_output.png,NaN


In [7]:
print(CNNSpot_fake.part0.unique())
print(CNNSpot_fake.part1.unique())
print(CNNSpot_fake.part2.unique())
print(CNNSpot_fake.part3.unique())

# ! train2 and the different generators from the test split follow different path structures
# training set: train2/<obejct_name>/0_real/<file_name>
# testing set: <generator_name>/1_fake/<file_name>

['stylegan' 'train2' 'cyclegan' 'imle' 'stylegan2' 'san' 'crn' 'gaugan'
 'biggan']
['car' 'person' 'orange' 'cow' '1_fake' 'airplane' 'church' 'dog' 'bird'
 'chair' 'bottle' 'sofa' 'bicycle' 'summer' 'pottedplant' 'winter' 'sheep'
 'horse' 'cat' 'diningtable' 'motorbike' 'train' 'boat' 'tvmonitor' 'bus'
 'bedroom' 'zebra' 'apple']
['1_fake' '00100639.png' '00100807.png' ... '00100825.png' '00267139.png'
 '000000043435.png']
['008017.png' '00570.png' 'n07740461_12101_fake.png' ... '036150.png'
 '095833.png' '075918.png']


In [8]:
CNNSpot_fake["real_path"] = CNNSpot_fake["path"]

mask_train = CNNSpot_fake["part0"] == "train2"
mask_not_train = CNNSpot_fake["part0"] != "train2"

CNNSpot_fake.loc[mask_train, "real_path"] = (
    CNNSpot_fake.loc[mask_train, "path"]
    .str.replace("^train2/", "CNNSpot/train/", regex=True)
)

CNNSpot_fake.loc[mask_not_train, "real_path"] = (
    "CNNSpot/test/" + CNNSpot_fake.loc[mask_not_train, "path"]
)

In [9]:
CNNSpot_fake

,path,dataset name,part0,part1,part2,part3,real_path
0,stylegan/car/1_fake/008017.png,CNNSpot,stylegan,car,1_fake,008017.png,CNNSpot/test/stylegan/car/1_fake/008017.png
1,train2/person/1_fake/00570.png,CNNSpot,train2,person,1_fake,00570.png,CNNSpot/train/person/1_fake/00570.png
3,cyclegan/orange/1_fake/n07740461_12101_fake.png,CNNSpot,cyclegan,orange,1_fake,n07740461_12101_fake.png,CNNSpot/test/cyclegan/orange/1_fake/n07740461_...
4,train2/cow/1_fake/01321.png,CNNSpot,train2,cow,1_fake,01321.png,CNNSpot/train/cow/1_fake/01321.png
5,cyclegan/orange/1_fake/n07740461_14960_fake.png,CNNSpot,cyclegan,orange,1_fake,n07740461_14960_fake.png,CNNSpot/test/cyclegan/orange/1_fake/n07740461_...
...,...,...,...,...,...,...,...
100694,stylegan2/cat/1_fake/001589.png,CNNSpot,stylegan2,cat,1_fake,001589.png,CNNSpot/test/stylegan2/cat/1_fake/001589.png
100695,stylegan2/horse/1_fake/001545.png,CNNSpot,stylegan2,horse,1_fake,001545.png,CNNSpot/test/stylegan2/horse/1_fake/001545.png
100696,train2/train/1_fake/00997.png,CNNSpot,train2,train,1_fake,00997.png,CNNSpot/train/train/1_fake/00997.png
100698,crn/1_fake/100212_output.png,CNNSpot,crn,1_fake,100212_output.png,NaN,CNNSpot/test/crn/1_fake/100212_output.png


In [10]:
root = Path("/work/tischuet/datasets/")

CNNSpot_fake["exists"] = [
    (root / p).exists() for p in CNNSpot_fake["real_path"]
]

print(CNNSpot_fake["exists"].value_counts()) # results mean that all file paths exist, within the CNNSpot folder

exists
True     71535
False     1055
Name: count, dtype: int64


In [11]:
pd.set_option("display.max_colwidth", None)

In [12]:
missing = CNNSpot_fake[~CNNSpot_fake["exists"]]

missing["real_path"].head(20)

21      CNNSpot/test/cyclegan/summer/1_fake/2016-01-02 20_15_00_fake.png
28      CNNSpot/test/cyclegan/winter/1_fake/2013-07-22 00_17_40_fake.png
472     CNNSpot/test/cyclegan/winter/1_fake/2016-05-21 00_48_51_fake.png
480     CNNSpot/test/cyclegan/winter/1_fake/2015-06-29 14_59_01_fake.png
495     CNNSpot/test/cyclegan/summer/1_fake/2009-02-20 00_18_51_fake.png
519     CNNSpot/test/cyclegan/winter/1_fake/2012-06-14 05_38_40_fake.png
547     CNNSpot/test/cyclegan/summer/1_fake/2013-03-02 03_57_51_fake.png
611     CNNSpot/test/cyclegan/winter/1_fake/2013-07-12 20_32_51_fake.png
655     CNNSpot/test/cyclegan/winter/1_fake/2013-08-23 20_48_11_fake.png
892     CNNSpot/test/cyclegan/winter/1_fake/2012-05-15 07_40_41_fake.png
893     CNNSpot/test/cyclegan/winter/1_fake/2011-10-12 01_13_31_fake.png
1101    CNNSpot/test/cyclegan/summer/1_fake/2016-01-12 20_58_51_fake.png
1260    CNNSpot/test/cyclegan/summer/1_fake/2010-11-04 16_00_00_fake.png
1484    CNNSpot/test/cyclegan/winter/1_fake/2015-09

In [13]:
len(CNNSpot_fake[
    CNNSpot_fake["part1"].isin(["summer", "winter"])
])

1055

In [13]:
mask = CNNSpot_fake["part1"].isin(["summer", "winter"])

CNNSpot_fake.loc[mask, "real_path"] = CNNSpot_fake.loc[mask, "real_path"].str.replace(
    r'(\d{2})_(\d{2})_(\d{2})(_[^/]+$)',
    r'\1:\2:\3\4',
    regex=True
)

In [14]:
root = Path("/work/tischuet/datasets/")

CNNSpot_fake["exists"] = [
    (root / p).exists() for p in CNNSpot_fake["real_path"]
]

print(CNNSpot_fake["exists"].value_counts()) # results mean that all file paths exist, within the CNNSpot folder

exists
True    72590
Name: count, dtype: int64


In [15]:
duplicates_CNN_fake = CNNSpot_fake.real_path[CNNSpot_fake.real_path.duplicated()]
(f"Number of duplicate paths: {len(duplicates_CNN_fake)}")

'Number of duplicate paths: 4471'

In [16]:
duplicates_CNN_fake

773                                CNNSpot/test/biggan/1_fake/00030232.png
1777                           CNNSpot/test/stylegan/car/1_fake/008182.png
2900                                 CNNSpot/test/imle/1_fake/00100012.png
3069                          CNNSpot/test/stylegan2/cat/1_fake/000207.png
3168                        CNNSpot/test/stylegan2/horse/1_fake/000209.png
                                        ...                               
100629    CNNSpot/test/cyclegan/winter/1_fake/2014-08-12 20:03:01_fake.png
100645    CNNSpot/test/cyclegan/summer/1_fake/2013-01-15 02:25:31_fake.png
100646                      CNNSpot/test/stylegan2/horse/1_fake/000168.png
100680                        CNNSpot/test/stylegan2/car/1_fake/000145.png
100698                           CNNSpot/test/crn/1_fake/100212_output.png
Name: real_path, Length: 4471, dtype: object

In [17]:
duplicates_CNN_fake = CNNSpot_fake.path[CNNSpot_fake.path.duplicated()]
(f"Number of duplicate paths: {len(duplicates_CNN_fake)}")

'Number of duplicate paths: 4471'

In [23]:
src_root = Path("/work/tischuet/datasets")      # where files currently are
dst_root = Path("/work/tischuet/replication_datasets")   # where you want them

for rel_path in CNNSpot_fake["real_path"]:
    src = src_root / rel_path
    dst = dst_root / rel_path

    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)  # create folders
        shutil.copy(src, dst)  # preserves metadata
    else:
        print(f"Missing: {src}")

In [22]:
root = Path("/work/tischuet/replication_datasets/CNNSpot")

extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}

count = sum(1 for p in root.rglob("*") if p.suffix.lower() in extensions)

print(count)
print(length_CNNSpot_real - 18 + length_CNNSpot_fake - 4471)

94189
94189


## GenImage

In [ ]:
count_images_recursive("../../datasets/GenImage/")

443386

In [30]:
print(count_images_recursive("../../datasets/GenImage/GenImage/ADM"))
print(count_images_recursive("../../datasets/GenImage/GenImage/BigGAN"))
print(count_images_recursive("../../datasets/GenImage/GenImage/glide"))
print(count_images_recursive("../../datasets/GenImage/GenImage/Midjourney"))
print(count_images_recursive("../../datasets/GenImage/GenImage/stable_diffusion_v_1_4"))
print(count_images_recursive("../../datasets/GenImage/GenImage/stable_diffusion_v_1_5"))
print(count_images_recursive("../../datasets/GenImage/GenImage/VQDM"))
print(count_images_recursive("../../datasets/GenImage/GenImage/wukong"))

46956
42981
35805
56509
57339
77770
53939
72087


In [35]:
random.seed(42)

def sample_equal_from_folders(src_folders, dst_root, n_per_folder=2500, move=False):
    dst_root = Path(dst_root)
    dst_root.mkdir(parents=True, exist_ok=True)

    for folder in src_folders:
        folder = Path(folder)

        # recursively get all files
        files = [f for f in folder.rglob("*") if f.is_file()]

        if len(files) < n_per_folder:
            raise ValueError(f"{folder} has fewer than {n_per_folder} files")

        sampled = random.sample(files, n_per_folder)

        for file in sampled:
            # preserve relative path
            rel_path = file.relative_to(folder)

            # recreate full structure inside destination
            dst_path = dst_root / folder.name / rel_path
            dst_path.parent.mkdir(parents=True, exist_ok=True)

            if move:
                shutil.move(file, dst_path)
            else:
                shutil.copy(file, dst_path)

In [5]:
def move_images(src_dir, dst_dir):
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    image_exts = (".png", ".jpg", ".jpeg", ".bmp", ".gif")

    for file in src_dir.iterdir():
        if file.is_file() and file.suffix.lower() in image_exts:
            shutil.move(str(file), dst_dir / file.name)

In [ ]:
""" move_images("/work/tischuet/datasets/GenImage/GenImage/wukong/imagenet_ai_0424_wukong/train/ai", "/work/tischuet/datasets/GenImage/GenImage/wukong")
move_images("/work/tischuet/datasets/GenImage/GenImage/VQDM/imagenet_ai_0419_vqdm/train/ai", "/work/tischuet/datasets/GenImage/GenImage/VQDM")
move_images("/work/tischuet/datasets/GenImage/GenImage/stable_diffusion_v_1_5/imagenet_ai_0424_sdv5/train/ai", "/work/tischuet/datasets/GenImage/GenImage/stable_diffusion_v_1_5")
move_images("/work/tischuet/datasets/GenImage/GenImage/stable_diffusion_v_1_4/imagenet_ai_0419_sdv4/train/ai", "/work/tischuet/datasets/GenImage/GenImage/stable_diffusion_v_1_4")
move_images("/work/tischuet/datasets/GenImage/GenImage/Midjourney/imagenet_midjourney/train/ai", "/work/tischuet/datasets/GenImage/GenImage/Midjourney")
move_images("/work/tischuet/datasets/GenImage/GenImage/glide/imagenet_glide/train/ai", "/work/tischuet/datasets/GenImage/GenImage/glide")
move_images("/work/tischuet/datasets/GenImage/GenImage/BigGAN/imagenet_ai_0419_biggan/train/ai", "/work/tischuet/datasets/GenImage/GenImage/BigGAN")
move_images("/work/tischuet/datasets/GenImage/GenImage/ADM/imagenet_ai_0508_adm/train/ai", "/work/tischuet/datasets/GenImage/GenImage/ADM") """

In [8]:
folders = [
    "/work/tischuet/datasets/GenImage/GenImage/ADM",
    "/work/tischuet/datasets/GenImage/GenImage/BigGAN",
    "/work/tischuet/datasets/GenImage/GenImage/glide",
    "/work/tischuet/datasets/GenImage/GenImage/Midjourney",
    "/work/tischuet/datasets/GenImage/GenImage/stable_diffusion_v_1_4",
    "/work/tischuet/datasets/GenImage/GenImage/stable_diffusion_v_1_5",
    "/work/tischuet/datasets/GenImage/GenImage/VQDM",
    "/work/tischuet/datasets/GenImage/GenImage/wukong",
]

sample_equal_from_folders(
    src_folders=folders,
    dst_root="../../replication_datasets/GenImage",
    n_per_folder=2500,
    move=False
)

In [11]:
count_images_recursive("../../replication_datasets/GenImage")

20000

## UDF

### Real

In [4]:
udf_real = test_real[test_real["dataset name"] == "Universal Fake Detect"].copy()

len(udf_real)

45616

In [ ]:
laion = count_images_in_folder("../../datasets/UDF/laion/0_real")
print(laion)

imagenet = count_images_in_folder("../../datasets/UDF/imagenet/0_real")
print(imagenet)

# only 100 images per folder -> resample them

1000
1000


### Fake

In [9]:
udf_fake = test_generated[test_generated["dataset name"] == "Universal Fake Detect"].copy()

len(udf_fake)

8890

In [16]:
duplicates_udf_fake = udf_fake.path[udf_fake.path.duplicated()]
(f"Number of duplicate paths: {len(duplicates_udf_fake)}")

'Number of duplicate paths: 1890'

In [17]:
duplicates_udf_fake

2441      glide_100_27/1_fake/apwufqzgrg.png
3262             dalle/1_fake/eoacigirrk.png
3645      glide_100_10/1_fake/drnfeijjlp.png
3766           ldm_100/1_fake/fihznganle.png
4774             dalle/1_fake/crwemtqnoq.png
                         ...                
100465    glide_100_27/1_fake/envczpiyds.png
100490          guided/1_fake/cpyrataump.png
100608           dalle/1_fake/gacytxnrbg.png
100625          guided/1_fake/gmjuidkgzm.png
100639          guided/1_fake/csvtttfwut.png
Name: path, Length: 1890, dtype: object

In [ ]:
root = Path("../../datasets/UDF/")

udf_fake["exists"] = [
    (root / p).exists() for p in udf_fake["path"]
]

print(udf_fake["exists"].value_counts()) # all paths exist

exists
True    8890
Name: count, dtype: int64


In [ ]:
dalle = count_images_in_folder("../../datasets/UDF/dalle/1_fake")
print(f"Image in dalle folder: {dalle}")

glide_50_27 = count_images_in_folder("../../datasets/UDF/glide_50_27/1_fake")
print(f"Image in dalle folder: {glide_50_27}")

glide_100_10 = count_images_in_folder("../../datasets/UDF/glide_100_10/1_fake")
print(f"Image in dalle folder: {glide_100_10}")

glide_100_27 = count_images_in_folder("../../datasets/UDF/glide_100_27/1_fake")
print(f"Image in dalle folder: {glide_100_27}")

guided = count_images_in_folder("../../datasets/UDF/guided/1_fake")
print(f"Image in dalle folder: {guided}")

ldm_100 = count_images_in_folder("../../datasets/UDF/ldm_100/1_fake")
print(f"Image in dalle folder: {ldm_100}")

ldm_200 = count_images_in_folder("../../datasets/UDF/ldm_200/1_fake")
print(f"Image in dalle folder: {ldm_200}")

ldm_200_cfg = count_images_in_folder("../../datasets/UDF/ldm_200_cfg/1_fake")
print(f"Image in dalle folder: {ldm_200_cfg}")

# ldm_200_cfg is not in excel sheet, but I take it anyways, as there are duplicates in the sheet and I can fill up the spots at least a little

Image in dalle folder: 1000
Image in dalle folder: 1000
Image in dalle folder: 1000
Image in dalle folder: 1000
Image in dalle folder: 1000
Image in dalle folder: 1000
Image in dalle folder: 1000
Image in dalle folder: 1000


In [25]:
flatten_with_path_names_fast(
    src_root="../../datasets/UDF",
    dst_root="../../replication_datasets/UDF",
    move=False,
    max_workers=128
)

Processing files: 100%|██████████| 8000/8000 [00:02<00:00, 3240.00file/s] 


In [26]:
count_images_recursive("../../replication_datasets/UDF")

8000

## CNNSpotset

### Real

In [3]:
set_real = test_real[test_real["dataset name"] == "CNNSpotset"].copy()
len(set_real)

30020

In [4]:
set_real

,path,dataset name
0,imle/0_real/00100162.png,CNNSpotset
1,progan/sofa/0_real/19497.png,CNNSpotset
8,stylegan2/cat/0_real/10123.png,CNNSpotset
11,progan/bus/0_real/14651.png,CNNSpotset
17,deepfake/0_real/420_0294.png,CNNSpotset
...,...,...
101694,stylegan2/church/0_real/07260.png,CNNSpotset
101699,whichfaceisreal/0_real/00328.jpeg,CNNSpotset
101702,stylegan/car/0_real/00501.png,CNNSpotset
101708,deepfake/0_real/115_0121.png,CNNSpotset


In [5]:
duplicates_set_real = set_real.path[set_real.path.duplicated()]
(f"Number of duplicate paths: {len(duplicates_set_real)}")

'Number of duplicate paths: 7218'

In [8]:
duplicates = set(set_real["path"]).intersection(set(CNNSpot_real["path"]))
len(duplicates) #no duplicates of real images

0

In [11]:
root = Path("/work/tischuet/datasets/CNNSpot/test")

set_real["exists"] = [
    (root / p).exists() for p in set_real["path"]
]

print(set_real["exists"].value_counts())

exists
True     29132
False      888
Name: count, dtype: int64


In [12]:
missing = set_real[~set_real["exists"]]

print(len(missing))
missing["path"].head(20)

888


28      cyclegan/winter/0_real/2008-07-09 20_32_51_rea...
148     cyclegan/summer/0_real/2015-07-15 00_39_20_rea...
222     cyclegan/summer/0_real/2015-07-26 18_54_20_rea...
310     cyclegan/winter/0_real/2017-03-05 17_15_40_rea...
326     cyclegan/summer/0_real/2014-07-19 04_26_01_rea...
389     cyclegan/winter/0_real/2016-02-04 14_41_51_rea...
463     cyclegan/summer/0_real/2016-09-01 00_00_00_rea...
500     cyclegan/summer/0_real/2014-09-14 20_39_50_rea...
541     cyclegan/winter/0_real/2016-12-29 15_19_00_rea...
645     cyclegan/summer/0_real/2013-08-28 06_26_41_rea...
772     cyclegan/winter/0_real/2014-03-08 12_26_51_rea...
781     cyclegan/summer/0_real/2011-06-03 03_36_41_rea...
848     cyclegan/summer/0_real/2013-08-21 04_54_41_rea...
877     cyclegan/summer/0_real/2011-07-06 16_55_20_rea...
999     cyclegan/winter/0_real/2017-03-03 09_20_41_rea...
1141    cyclegan/summer/0_real/2013-09-14 12_02_11_rea...
1250    cyclegan/summer/0_real/2015-07-15 11_58_01_rea...
1254    cycleg

In [13]:
mask = set_real["path"].str.contains("summer|winter", case=False, na=False)

set_real.loc[mask, "path"] = set_real.loc[mask, "path"].str.replace(
    r'(\d{2})_(\d{2})_(\d{2})(_[^/]+$)',
    r'\1:\2:\3\4',
    regex=True
)

In [15]:
root = Path("/work/tischuet/datasets/CNNSpot/test")

set_real["exists"] = [
    (root / p).exists() for p in set_real["path"]
]

print(set_real["exists"].value_counts())

exists
True    30020
Name: count, dtype: int64


In [19]:
count_images_recursive("/work/tischuet/replication_datasets/CNNSpot")

94189

In [21]:
src_root = Path("/work/tischuet/datasets/CNNSpot/test")      # where files currently are
dst_root = Path("/work/tischuet/replication_datasets/CNNSpotset")   # where you want them

for rel_path in set_real["path"]:
    src = src_root / rel_path
    dst = dst_root / rel_path

    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)  # create folders
        shutil.copy(src, dst)  # doesn't preserve metadata
    else:
        print(f"Missing: {src}")

In [22]:
print(count_images_recursive("/work/tischuet/replication_datasets/CNNSpotset"))

print(30020-7218)

22802
22802


In [58]:
flatten_with_path_names_fast(
    src_root="/work/tischuet/replication",
    dst_root="../../replication_datasets/CNNSpot",
    move=False,
    max_workers=128
)

Processing files: 100%|██████████| 22802/22802 [00:18<00:00, 1211.53file/s]


In [59]:
print(count_images_recursive("/work/tischuet/replication_datasets/CNNSpot"))
print(94189+22802)

116991
116991


## Imagenet Val

In [2]:
from datasets import load_dataset
from pathlib import Path

out_dir = Path("/work/tischuet/datasets/Imagetnet_val")
out_dir.mkdir(parents=True, exist_ok=True)

# Stream the validation split instead of reading remote parquet with Dask
ds = load_dataset(
    "ILSVRC/imagenet-1k",
    split="validation",
    streaming=True
)

# Shuffle the stream first, then take 2000 examples
sample = ds.shuffle(seed=42, buffer_size=50_000).take(2000)

for i, example in enumerate(sample):
    img = example["image"]          # already a PIL image for Image columns
    label = example.get("label", -1)
    img.convert("RGB").save(out_dir / f"{i:04d}_class{label}.png")

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

## Helper functions

In [2]:
from pathlib import Path
import shutil
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import os

def flatten_with_path_names_fast(
    src_root,
    dst_root,
    move=False,
    sep="__",
    max_workers=8,
    prefix=None,              # "0_real", "1_fake", or None
    prefix_sep="__"           # separator between prefix and filename
):
    src_root = Path(src_root).resolve()
    dst_root = Path(dst_root).resolve()

    if src_root == dst_root:
        raise ValueError("src_root and dst_root must be different")

    dst_root.mkdir(parents=True, exist_ok=True)

    # Collect files
    all_files = []
    for root, _, files in os.walk(src_root):
        root = Path(root)
        for name in files:
            all_files.append(root / name)

    def process_file(path):
        rel_path = path.relative_to(src_root)

        # flatten path into filename
        base_name = sep.join(rel_path.parts)

        # optionally add prefix
        if prefix is not None:
            new_name = f"{prefix}{prefix_sep}{base_name}"
        else:
            new_name = base_name

        dst_path = dst_root / new_name

        # Handle collisions
        if dst_path.exists():
            stem = dst_path.stem
            suffix = dst_path.suffix
            i = 1
            while True:
                candidate = dst_root / f"{stem}_{i}{suffix}"
                if not candidate.exists():
                    dst_path = candidate
                    break
                i += 1

        if move:
            shutil.move(str(path), str(dst_path))
        else:
            shutil.copy(str(path), str(dst_path))

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_file, path) for path in all_files]
        for _ in tqdm(as_completed(futures), total=len(futures), desc="Processing files", unit="file"):
            pass

In [1]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def remove_pngs_parallel(folder_path, max_workers=8):
    folder = Path(folder_path)

    png_files = [
        file for file in folder.iterdir()
        if file.is_file() and file.suffix.lower() == ".png"
    ]

    def delete_file(file_path):
        file_path.unlink()
        return file_path

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(delete_file, file) for file in png_files]

        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Deleting PNGs",
            unit="file"
        ):
            future.result()

remove_pngs_parallel("../../replication_datasets/CNNSpot", max_workers=16)

Deleting PNGs: 100%|██████████| 12875/12875 [02:05<00:00, 102.92file/s]


In [4]:
# count image files in folder without recursion
def count_images_in_folder(folder_path):
    IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

    folder = Path(folder_path)

    count = sum(
        1 for file in folder.iterdir()
        if file.is_file() and file.suffix.lower() in IMAGE_EXTS
    )

    return count


# count image files recursively (including all subfolders)
def count_images_recursive(folder_path):
    IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

    folder = Path(folder_path)

    count = sum(
        1 for file in folder.rglob("*")
        if file.is_file() and file.suffix.lower() in IMAGE_EXTS
    )

    return count

In [ ]:
source = count_images_in_folder("/work/tischuet/replication_datasets/CNNSpot/test")
source_2 = count_images_in_folder("/work/tischuet/replication_datasets/CNNSpot/train")

root = count_images_in_folder("/work/tischuet/replication_datasets/CNNSpot_flat")


print(source+source_2)
print(root)

0
94189


In [6]:
print(count_images_in_folder("/work/tischuet/replication_datasets/UDF"))
print(count_images_in_folder("/work/tischuet/replication_datasets/GenImage"))
print(count_images_in_folder("/work/tischuet/replication_datasets/CNNSpot"))

8000
20000
116991


In [8]:
# Path to your folder
folder_path = "/work/tischuet/replication_datasets/CNNSpot"

# Supported image extensions
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp")

# List all image files
image_files = [
    f for f in os.listdir(folder_path)
    if f.lower().endswith(image_extensions)
]


real = 0
fake = 0
other = 0
# Check filenames
for filename in image_files:
    if "0_real" in filename:
        real += 1
    elif "1_fake" in filename:
        fake += 1
    else:
        other +=1
    
print(real)
print(fake)
print(other)
print(fake+real)

48872
68119
0
116991


In [9]:
print(f"Real images ot sample: {96119-48872}")

Real images ot sample: 47247


In [40]:
# sample real images

sample_equal_from_folders(
    src_folders=["/work/tischuet/datasets/MS_COCO_train2017"],
    dst_root="../../replication_datasets/MS_COCO_train2017",
    n_per_folder=47247,
    move=False
)

In [42]:
count_images_in_folder("/work/tischuet/replication_datasets/MS_COCO_train2017/MS_COCO_train2017")

47247